In [1]:
!pip install transformers
!pip install datasets evaluate accelerate sentencepiece sacrebleu rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.8 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=4f6c1a31c2d2d8cef94ded9fb77a4f11cde692c00a189b4b56b99ab91deb98f1
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [2]:
## Loading the dataset
from datasets import load_dataset
spider = load_dataset("xlangai/spider")

print(spider)
print(spider["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/5.51k [00:00<?, ?B/s]

spider/train-00000-of-00001.parquet:   0%|          | 0.00/831k [00:00<?, ?B/s]

spider/validation-00000-of-00001.parquet:   0%|          | 0.00/126k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1034 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
        num_rows: 7000
    })
    validation: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
        num_rows: 1034
    })
})
{'db_id': 'department_management', 'query': 'SELECT count(*) FROM head WHERE age  >  56', 'question': 'How many heads of the departments are older than 56 ?', 'query_toks': ['SELECT', 'count', '(', '*', ')', 'FROM', 'head', 'WHERE', 'age', '>', '56'], 'query_toks_no_value': ['select', 'count', '(', '*', ')', 'from', 'head', 'where', 'age', '>', 'value'], 'question_toks': ['How', 'many', 'heads', 'of', 'the', 'departments', 'are', 'older', 'than', '56', '?']}


In [3]:
#Loading table schema json
import json

with open("/content/tables.json","r") as f:
  tables_data=json.load(f)

In [4]:
print(tables_data[0].keys())
print(tables_data[0])

dict_keys(['column_names', 'column_names_original', 'column_types', 'db_id', 'foreign_keys', 'primary_keys', 'table_names', 'table_names_original'])
{'column_names': [[-1, '*'], [0, 'perpetrator id'], [0, 'people id'], [0, 'date'], [0, 'year'], [0, 'location'], [0, 'country'], [0, 'killed'], [0, 'injured'], [1, 'people id'], [1, 'name'], [1, 'height'], [1, 'weight'], [1, 'home town']], 'column_names_original': [[-1, '*'], [0, 'Perpetrator_ID'], [0, 'People_ID'], [0, 'Date'], [0, 'Year'], [0, 'Location'], [0, 'Country'], [0, 'Killed'], [0, 'Injured'], [1, 'People_ID'], [1, 'Name'], [1, 'Height'], [1, 'Weight'], [1, 'Home Town']], 'column_types': ['text', 'number', 'number', 'text', 'number', 'text', 'text', 'number', 'number', 'number', 'text', 'number', 'number', 'text'], 'db_id': 'perpetrator', 'foreign_keys': [[2, 9]], 'primary_keys': [1, 9], 'table_names': ['perpetrator', 'people'], 'table_names_original': ['perpetrator', 'people']}


In [5]:
# build schema
def build_schema_text(db_schema):
    table_names = db_schema["table_names_original"]
    column_names = db_schema["column_names_original"]

    # Create empty column list for each table
    table_to_columns = {
        table_name: []
        for table_name in table_names
    }



    # Populate column list for each table

    # column_names_original format:
    # [-1, "*"] means ignore
    # [0, "department_id"] means table index 0 has column department_id
    for table_id, column_name in column_names:
        if table_id == -1:
            continue

        table_name = table_names[table_id]
        table_to_columns[table_name].append(column_name)

    # Convert to text format:
    # department(department_id, name); head(head_id, age)
    schema_parts = []

    for table_name, columns in table_to_columns.items():
        columns_text = ", ".join(columns)
        schema_parts.append(f"{table_name}({columns_text})")

    return "; ".join(schema_parts)


In [6]:
schema_map = {}

for db_schema in tables_data:
    db_id = db_schema["db_id"]
    schema_map[db_id] = build_schema_text(db_schema)

print("Total schemas:", len(schema_map))


Total schemas: 166


In [7]:
# Convert Spider dataset into training format
# { input_text:"text2sql:database:schema question:get employee from employee", "target_text":"select * from employees" }
def convert_spider_example(example):
  schema_text = schema_map.get(example["db_id"], "")
  return{
      "input_text": ("text2sql:"
      f"question:{example['question']} "
      f"database:{example['db_id']} "
      f"schema: {schema_text} "
      ),
      "target_text": example["query"]
  }


In [8]:

spider_text2sql=spider.map(convert_spider_example)

print(spider_text2sql["train"][0]["input_text"])
print(spider_text2sql["train"][0]["target_text"])

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1034 [00:00<?, ? examples/s]

text2sql:question:How many heads of the departments are older than 56 ? database:department_management schema: department(Department_ID, Name, Creation, Ranking, Budget_in_Billions, Num_Employees); head(head_ID, name, born_state, age); management(department_ID, head_ID, temporary_acting) 
SELECT count(*) FROM head WHERE age  >  56


In [9]:
#Train & Validation split
train=spider_text2sql["train"].shuffle(seed=42).select(range(3000))
eval_data=spider_text2sql["validation"].shuffle(seed=42).select(range(100))

In [10]:
train

Dataset({
    features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks', 'input_text', 'target_text'],
    num_rows: 3000
})

In [11]:
#Load t5
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


MODEL_NAME = "t5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Device:", device)



config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Device: cuda


In [12]:
MAX_INPUT_LENGTH=512
MAX_OUTPUT_LENGTH=128

def preporcess_function(examples):
  model_inputs=tokenizer( examples["input_text"],
                         max_length=MAX_INPUT_LENGTH,
                         truncation=True)
  labels=tokenizer(text_target=examples["target_text"],
                   max_length=MAX_OUTPUT_LENGTH,
                   truncation=True)
  model_inputs["labels"]=labels["input_ids"]
  return model_inputs

In [13]:
tokenized_train=train.map(preporcess_function, batched=True,remove_columns=train.column_names)
tokenized_eval=eval_data.map(preporcess_function, batched=True,remove_columns=eval_data.column_names)

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [14]:
## Sample pre test of model before training
def generate_sql(prompt):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LENGTH
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_length=128,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)
prompt = "text2sql: database: department_management question: How many heads of the departments are older than 56?"

print(generate_sql(prompt))

not_entailment


In [15]:
import torch
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-spider-text2sql",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=4,
    weight_decay=0.01,
    predict_with_generate=True,
    logging_steps=50,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator
)

trainer.train()
trainer.save_model("./t5-spider-finetuned")
tokenizer.save_pretrained("./t5-spider-finetuned")




Epoch,Training Loss,Validation Loss
1,0.501224,0.594394
2,0.376275,0.527928
3,0.307810,0.518459


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,0.501224,0.594394
2,0.376275,0.527928
3,0.307810,0.518459
4,0.270223,0.514223


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./t5-spider-finetuned/tokenizer_config.json',
 './t5-spider-finetuned/tokenizer.json')

In [16]:
## generate sql after fine tuned method.
def generate_finetuned_sql(prompt, max_length=128):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)
prompt = "text2sql: database: department_management question: How many heads of the departments are older than 56?"

print(generate_finetuned_sql(prompt))

SELECT count(*) FROM departments WHERE he > 56


In [17]:
# Evaluting the trained Modal
import numpy as np
import pandas as pd
import re


In [29]:
import re

# Normalizing the SQL
def normalize_sql(sql):
    sql = sql.lower()
    sql = re.sub(r"\s+", " ", sql)
    sql = sql.strip()
    return sql

def generate_prediction_batch(dataset, max_samples=100):
  rows=[]
  for i in range(min(max_samples, len(dataset))):
    prompt=dataset[i]["input_text"]
    db_id=dataset[i]["db_id"]
    question=dataset[i]["question"]
    gold_sql=dataset[i]["target_text"]

    predicted_sql=generate_finetuned_sql(prompt)

    rows.append({
            "input_text": prompt,
            "gold_sql": gold_sql,
            "predicted_sql": predicted_sql,
            "exact_match": normalize_sql(gold_sql) == normalize_sql(predicted_sql)
        })

  return pd.DataFrame(rows)

In [30]:
eval_results_df=generate_prediction_batch(eval_data, max_samples=100)
exact_match_score = eval_results_df["exact_match"].mean()

print("Exact Match:", exact_match_score)

eval_results_df.head(10)

Exact Match: 0.18


,input_text,gold_sql,predicted_sql,exact_match
0,text2sql:question:How many players are from ea...,"SELECT count(*) , country_code FROM players G...","SELECT count(*) , country_code FROM players GR...",True
1,text2sql:question:How many documents are using...,SELECT count(*) FROM Documents AS T1 JOIN Temp...,SELECT count(*) FROM Documents AS T1 JOIN Temp...,True
2,text2sql:question:How many cartoons were writt...,SELECT count(*) FROM Cartoon WHERE Written_by ...,SELECT count(*) FROM Cartoon WHERE Written_by ...,False
3,"text2sql:question:Show name, country, age for ...","SELECT name , country , age FROM singer ORDE...","SELECT Name , Country , Age FROM singer ORDER ...",True
4,text2sql:question:What are the names of the co...,"SELECT Name FROM country WHERE continent = ""...","SELECT Name FROM country WHERE Continent = ""Eu...",False
5,text2sql:question:How many high schoolers are ...,SELECT count(*) FROM Highschooler WHERE grade ...,SELECT count(*) FROM Highschooler WHERE grade ...,True
6,text2sql:question:How many countries speak bot...,SELECT COUNT(*) FROM (SELECT T1.Name FROM coun...,SELECT count(*) FROM country AS T1 JOIN countr...,False
7,text2sql:question:List the name of teachers wh...,"select name from teacher where hometown != ""li...",SELECT Name FROM teacher WHERE Hometown != ''L...,False
8,"text2sql:question:What is the id, line 1, and ...","SELECT T1.address_id , T1.line_1 , T1.line_2...","SELECT T1.address_id , T2.line_1 , T2.line_2 F...",False
9,text2sql:question:What are the orchestras that...,SELECT Orchestra FROM orchestra WHERE Orchestr...,SELECT orchestra FROM performance GROUP BY orc...,False


In [31]:
correct_predictions = eval_results_df[eval_results_df["exact_match"] == True]

correct_predictions[["input_text", "gold_sql", "predicted_sql"]]

,input_text,gold_sql,predicted_sql
0,text2sql:question:How many players are from ea...,"SELECT count(*) , country_code FROM players G...","SELECT count(*) , country_code FROM players GR..."
1,text2sql:question:How many documents are using...,SELECT count(*) FROM Documents AS T1 JOIN Temp...,SELECT count(*) FROM Documents AS T1 JOIN Temp...
3,"text2sql:question:Show name, country, age for ...","SELECT name , country , age FROM singer ORDE...","SELECT Name , Country , Age FROM singer ORDER ..."
5,text2sql:question:How many high schoolers are ...,SELECT count(*) FROM Highschooler WHERE grade ...,SELECT count(*) FROM Highschooler WHERE grade ...
11,text2sql:question:How many conductors are ther...,SELECT count(*) FROM conductor,SELECT count(*) FROM conductor
19,text2sql:question:List the section_name in rev...,SELECT section_name FROM Sections ORDER BY sec...,SELECT section_name FROM sections ORDER BY sec...
29,text2sql:question:How many teachers are there?...,SELECT count(*) FROM teacher,SELECT count(*) FROM teacher
32,text2sql:question:List the names of people tha...,SELECT Name FROM people WHERE People_ID NOT IN...,SELECT Name FROM people WHERE People_ID NOT IN...
35,text2sql:question:Return the type code of the ...,SELECT template_type_code FROM Ref_template_ty...,SELECT Template_Type_Code FROM Ref_Template_Ty...
43,text2sql:question:Find the number of pets whos...,SELECT count(*) FROM pets WHERE weight > 10,SELECT count(*) FROM Pets WHERE weight > 10


In [32]:
wrong_predictions = eval_results_df[eval_results_df["exact_match"] == False]

wrong_predictions[["input_text", "gold_sql", "predicted_sql"]].head(20)

,input_text,gold_sql,predicted_sql
2,text2sql:question:How many cartoons were writt...,SELECT count(*) FROM Cartoon WHERE Written_by ...,SELECT count(*) FROM Cartoon WHERE Written_by ...
4,text2sql:question:What are the names of the co...,"SELECT Name FROM country WHERE continent = ""...","SELECT Name FROM country WHERE Continent = ""Eu..."
6,text2sql:question:How many countries speak bot...,SELECT COUNT(*) FROM (SELECT T1.Name FROM coun...,SELECT count(*) FROM country AS T1 JOIN countr...
7,text2sql:question:List the name of teachers wh...,"select name from teacher where hometown != ""li...",SELECT Name FROM teacher WHERE Hometown != ''L...
8,"text2sql:question:What is the id, line 1, and ...","SELECT T1.address_id , T1.line_1 , T1.line_2...","SELECT T1.address_id , T2.line_1 , T2.line_2 F..."
9,text2sql:question:What are the orchestras that...,SELECT Orchestra FROM orchestra WHERE Orchestr...,SELECT orchestra FROM performance GROUP BY orc...
10,text2sql:question:Which models are lighter tha...,SELECT DISTINCT T1.model FROM MODEL_LIST AS T1...,SELECT model FROM model_list WHERE weight > 35...
12,text2sql:question:What is the name and capacit...,"select t2.name , t2.capacity from concert as ...","SELECT T1.name , T1.capacity FROM stadium AS T..."
13,text2sql:question:Return the grade for the hig...,SELECT grade FROM Highschooler WHERE name = ...,SELECT t1.grade FROM Highschooler AS t1 JOIN F...
14,text2sql:question:Find the semester when both ...,SELECT DISTINCT T2.semester_id FROM Degree_Pro...,SELECT semester_name FROM students WHERE semes...


In [33]:
def looks_like_sql(sql):
    sql = sql.lower().strip()
    return sql.startswith("select") and "from" in sql

eval_results_df["looks_like_sql"] = eval_results_df["predicted_sql"].apply(looks_like_sql)

print("Exact Match:", eval_results_df["exact_match"].mean())
print("Looks Like SQL:", eval_results_df["looks_like_sql"].mean())

Exact Match: 0.18
Looks Like SQL: 1.0
